# 💼 Financial Statement Agent
### Kaggle Capstone · 5-Day AI Agents Intensive Vibe Coding Course

**Track:** Agents for Business  
**Problem:** Small and medium businesses spend hours manually entering bank statement data into spreadsheets to build financial models. A bookkeeper processes 100–300 transactions per month, categorizing each one by hand — errors, delays, zero analytics.

**Solution:** A multi-agent pipeline powered by Gemini that:
1. **Parses** any bank statement (CSV / Excel / 1C format) autonomously
2. **Categorizes** transactions using a cascade: rules → keywords → Gemini AI
3. **Remembers** learned patterns (memory: counterparty → category rules)
4. **Builds** a financial model with P&L, cash flow, KPIs, NPV/IRR
5. **Delivers insights** with industry benchmarks and recommendations

**Course concepts applied (5 of 5):**
| Concept | Implementation |
|---|---|
| ✅ Tool use | Parser, categorizer, model builder as agent tools |
| ✅ Multi-agent | 4 specialized agents with handoff |
| ✅ Memory | Learned rules persist across agent calls |
| ✅ Evals & guardrails | Confidence scoring, low-confidence flagging |
| ✅ Production-ready | Deployed on Vercel + Neon Postgres |

**Repository:** https://github.com/Olegmeln/data_fin_model  
**Live demo:** https://data-fin-model.vercel.app

## 0. Setup

In [ ]:
!pip install -q google-generativeai pandas openpyxl matplotlib

In [ ]:
import os
import json
import csv
import io
import re
from dataclasses import dataclass, field, asdict
from datetime import date, datetime
from decimal import Decimal
from collections import defaultdict
from typing import Optional

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import google.generativeai as genai

# ── API key ──────────────────────────────────────────────────────────────────
# Kaggle: Add secret GEMINI_API_KEY via Add-ons → Secrets
from kaggle_secrets import UserSecretsClient
try:
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    print("⚠️  GEMINI_API_KEY not set — AI categorization will be skipped, rules+keywords only.")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini API configured")

## 1. Shared Data Structures

In [ ]:
@dataclass
class ParsedOperation:
    """Single bank transaction after parsing."""
    date: date
    amount: float          # always positive
    direction: str         # 'in' | 'out'
    counterparty: str
    description: str

@dataclass
class CategorizedOperation:
    """Parsed operation enriched with category and confidence."""
    op: ParsedOperation
    category_code: str
    category_name: str
    kind: str              # 'income' | 'expense' | 'transfer' | 'investing'
    confidence: float
    source: str            # 'rule' | 'keyword' | 'gemini' | 'fallback'
    needs_review: bool = False

# ── Financial model categories (from data_fin_model template) ────────────────
DEFAULT_CATEGORIES = [
    {"code": "REV_MAIN",   "name": "Выручка от продаж",               "kind": "income",
     "keywords": ["оплата по счет", "оплата по счёт", "оплата по договор", "за услуги",
                  "за товар", "аванс по договор", "выручк", "оплата заказа", "предоплата"]},
    {"code": "REV_OTHER",  "name": "Прочие поступления",              "kind": "income",
     "keywords": ["возврат", "процент", "кэшбэк", "кешбэк", "субсиди", "грант"]},
    {"code": "COGS",       "name": "Закупка товаров и материалов",    "kind": "expense",
     "keywords": ["закупк", "поставка товар", "материал", "сырь", "комплектующ"]},
    {"code": "PAYROLL",    "name": "Фонд оплаты труда",               "kind": "expense",
     "keywords": ["зарплат", "заработн", "аванс сотрудник", "отпускн",
                  "оплата труда", "премия сотрудник", "по реестру"]},
    {"code": "TAXES",      "name": "Налоги и взносы",                 "kind": "expense",
     "keywords": ["налог", "ндфл", "усн", "страхов", "взнос", "ифнс", "фнс",
                  "пфр", "фсс", "сфр", "патент", "енс", "единый налоговый"]},
    {"code": "RENT",       "name": "Аренда и коммунальные платежи",   "kind": "expense",
     "keywords": ["аренд", "коммунальн", "электроэнерг", "клининг"]},
    {"code": "MARKETING",  "name": "Маркетинг и реклама",             "kind": "expense",
     "keywords": ["реклам", "маркетинг", "директ", "google ads", "таргет",
                  "продвижени", "smm", "блогер", "лидогенерац"]},
    {"code": "SERVICES",   "name": "Сервисы и подрядчики",            "kind": "expense",
     "keywords": ["подписк", "хостинг", "лицензи", "saas", "облачн", "crm",
                  "консультацион", "бухгалтерск", "юридическ", "аутсорс"]},
    {"code": "LOGISTICS",  "name": "Логистика и доставка",            "kind": "expense",
     "keywords": ["доставк", "логистик", "транспортн", "сдэк", "cdek",
                  "boxberry", "почта росси", "курьер", "грузоперевозк"]},
    {"code": "BANK",       "name": "Банковские услуги",               "kind": "expense",
     "keywords": ["комисси", "обслуживание счет", "обслуживание счёт", "рко", "эквайринг"]},
    {"code": "OWNER",      "name": "Выплаты собственнику",            "kind": "expense",
     "keywords": ["дивиденд", "вывод средств", "выплата учредител", "под отчет", "подотчет"]},
    {"code": "CAPEX",      "name": "Инвестиции (CAPEX)",              "kind": "investing",
     "keywords": ["оборудован", "основное средств", "капитальн", "внеоборотн"]},
    {"code": "LOAN_IN",    "name": "Кредит / заём (поступление)",     "kind": "financing",
     "keywords": ["кредит", "займ", "ссуда", "финансирован"]},
    {"code": "LOAN_OUT",   "name": "Погашение кредита",               "kind": "financing",
     "keywords": ["погашен", "основной долг", "возврат займ", "выплата по кредит"]},
    {"code": "TRANSFER",   "name": "Внутренний перевод",              "kind": "transfer",
     "keywords": ["перевод между счет", "внутренний перевод", "собственный счет"]},
    {"code": "OTHER_EXP",  "name": "Прочие расходы",                  "kind": "expense",
     "keywords": []},
]

CAT_BY_CODE = {c["code"]: c for c in DEFAULT_CATEGORIES}
CONFIDENCE_THRESHOLD = 0.75   # below this → needs_review = True

print(f"✅ {len(DEFAULT_CATEGORIES)} financial categories loaded")

## 2. Agent 1 — Parsing Agent
> **Tool use:** The agent calls `parse_csv` or `parse_xlsx` as tools and auto-detects format.

In [ ]:
# ── Parsing utilities (adapted from data_fin_model/backend/app/parsers) ──────

_DATE_FORMATS = ("%d.%m.%Y", "%Y-%m-%d", "%d/%m/%Y", "%d.%m.%y", "%Y.%m.%d")
_OUT_MARKERS  = ("спис", "расход", "debit",  "оплата", "выдач", "списание")
_IN_MARKERS   = ("поступ", "приход", "credit", "зачис", "пополнен", "поступление")
_HEADER_HINTS = [
    ("date",         ("дата", "date")),
    ("debit",        ("расход", "списан", "дебет", "debit")),
    ("credit",       ("приход", "поступ", "кредит", "credit")),
    ("optype",       ("тип", "вид операции", "операция", "дт/кт")),
    ("counterparty", ("контрагент", "плательщик", "получатель", "корреспондент")),
    ("description",  ("назначен", "описан", "основание", "коммент", "purpose")),
    ("amount",       ("сумма", "amount", "итого")),
]

def _parse_date(value) -> Optional[date]:
    if isinstance(value, (date, datetime)):
        return value.date() if isinstance(value, datetime) else value
    s = str(value or "").strip().split(" ")[0].split("T")[0]
    for fmt in _DATE_FORMATS:
        try: return datetime.strptime(s, fmt).date()
        except ValueError: pass
    return None

def _parse_amount(value) -> Optional[float]:
    if value is None: return None
    if isinstance(value, (int, float)): return abs(float(value)) if float(value) != 0 else None
    s = re.sub(r"[^\d.\-,]", "", str(value).replace(" ", "").replace("\xa0", "")).replace(",", ".")
    try: v = float(s); return abs(v) if v != 0 else None
    except: return None

def _map_columns(headers: list) -> dict:
    low = [str(h or "").lower() for h in headers]
    mapping, used = {}, set()
    for role, hints in _HEADER_HINTS:
        for idx, h in enumerate(low):
            if idx in used or not h: continue
            if any(hint in h for hint in hints):
                mapping[role] = idx; used.add(idx); break
    return mapping

def _build_op(cells: list, cols: dict) -> Optional[ParsedOperation]:
    def cell(role): idx = cols.get(role); return cells[idx] if idx is not None and idx < len(cells) else None
    d = _parse_date(cell("date"))
    if d is None: return None
    amount, direction = None, None
    debit  = _parse_amount(cell("debit"))  if "debit"  in cols else None
    credit = _parse_amount(cell("credit")) if "credit" in cols else None
    if debit:  amount, direction = debit, "out"
    elif credit: amount, direction = credit, "in"
    if amount is None and "amount" in cols:
        v = _parse_amount(cell("amount"))
        if v is None: return None
        op_type = str(cell("optype") or "").lower()
        if any(m in op_type for m in _OUT_MARKERS): direction = "out"
        elif any(m in op_type for m in _IN_MARKERS): direction = "in"
        else: direction = "out"  # conservative default
        amount = v
    if not amount or not direction: return None
    return ParsedOperation(
        date=d, amount=amount, direction=direction,
        counterparty=str(cell("counterparty") or "").strip()[:200],
        description=str(cell("description") or "").strip(),
    )

# ── Tool: parse_csv ───────────────────────────────────────────────────────────
def parse_csv(raw: bytes) -> list[ParsedOperation]:
    """Agent tool: parse bank statement CSV, auto-detect encoding and delimiter."""
    for enc in ("utf-8-sig", "cp1251", "utf-8"):
        try: text = raw.decode(enc); break
        except: pass
    else: text = raw.decode("utf-8", errors="replace")
    delim = ";" if text.count(";") >= text.count(",") else ","
    rows = [r for r in csv.reader(io.StringIO(text), delimiter=delim) if any(c.strip() for c in r)]
    header_idx = next(
        (i for i, r in enumerate(rows[:10]) if "дата" in " ".join(r).lower() or "date" in " ".join(r).lower()),
        None
    )
    if header_idx is None: raise ValueError("Header row with 'Дата' not found")
    cols = _map_columns(rows[header_idx])
    return [op for row in rows[header_idx+1:] if (op := _build_op(row, cols)) is not None]

# ── Tool: parse_xlsx ──────────────────────────────────────────────────────────
def parse_xlsx(raw: bytes) -> list[ParsedOperation]:
    """Agent tool: parse bank statement Excel file."""
    df = pd.read_excel(io.BytesIO(raw), header=None, dtype=str)
    header_idx = next(
        (i for i, row in df.iterrows() if "дата" in " ".join(str(v) for v in row).lower()),
        None
    )
    if header_idx is None: raise ValueError("Header row not found in Excel")
    cols = _map_columns(list(df.iloc[header_idx]))
    return [
        op for _, row in df.iloc[header_idx+1:].iterrows()
        if (op := _build_op(list(row), cols)) is not None
    ]

# ── Agent 1: Parsing Agent ────────────────────────────────────────────────────
def parsing_agent(file_path: str) -> list[ParsedOperation]:
    """
    Agent 1 — Parsing Agent.
    Auto-detects file format and calls the appropriate parser tool.
    Returns a list of parsed operations.
    """
    print(f"\n🔍 [Parsing Agent] Processing: {file_path}")
    with open(file_path, "rb") as f:
        raw = f.read()
    ext = file_path.lower().split(".")[-1]
    if ext in ("xlsx", "xls"):
        ops = parse_xlsx(raw)
        fmt = "Excel"
    else:
        ops = parse_csv(raw)
        fmt = "CSV"
    print(f"   ✅ Format detected: {fmt} | Parsed: {len(ops)} operations")
    if ops:
        dates = [o.date for o in ops]
        print(f"   📅 Period: {min(dates)} → {max(dates)}")
        total_in  = sum(o.amount for o in ops if o.direction == "in")
        total_out = sum(o.amount for o in ops if o.direction == "out")
        print(f"   💰 Inflows: {total_in:,.0f} | Outflows: {total_out:,.0f}")
    return ops

## 3. Agent 2 — Categorization Agent
> **Multi-agent + Memory:** Uses a 4-level cascade. Learned rules (memory) take priority over AI.

In [ ]:
# ── Agent Memory: learned rules (counterparty → category) ────────────────────
# This is the agent's long-term memory within a session.
# In production (data_fin_model backend) these rules persist in PostgreSQL.
learned_rules: dict[str, str] = {}   # {counterparty_lower: category_code}

def teach_rule(counterparty: str, category_code: str):
    """Memory write: agent learns a new rule from user confirmation."""
    key = counterparty.strip().lower()
    if len(key) >= 4 and category_code in CAT_BY_CODE:
        learned_rules[key] = category_code
        print(f"   🧠 Rule learned: '{counterparty}' → {category_code}")

# ── Level 1: Memory — learned rules ──────────────────────────────────────────
def _apply_rules(op: ParsedOperation) -> Optional[tuple]:
    cp = op.counterparty.lower()
    ds = op.description.lower()
    for pattern, code in learned_rules.items():
        if pattern in cp or pattern in ds:
            return code, 0.97, "rule"
    return None

# ── Level 2: Built-in keywords ────────────────────────────────────────────────
def _apply_keywords(op: ParsedOperation) -> Optional[tuple]:
    cp = op.counterparty.lower()
    ds = op.description.lower()
    for cat in DEFAULT_CATEGORIES:
        kind_ok = (
            cat["kind"] == "transfer"
            or (cat["kind"] == "income"    and op.direction == "in")
            or (cat["kind"] != "income"    and op.direction == "out")
        )
        if kind_ok:
            for kw in cat["keywords"]:
                if kw in cp or kw in ds:
                    return cat["code"], 0.85, "keyword"
    return None

# ── Level 3: Gemini AI categorization ────────────────────────────────────────
def _ai_categorize_batch(items: list[dict]) -> dict[int, tuple]:
    """Call Gemini to categorize a batch of operations. Returns {idx: (code, confidence)}."""
    if not GEMINI_API_KEY or not items:
        return {}
    categories_block = "\n".join(
        f'- {c["code"]}: {c["name"]} ({c["kind"]})'
        for c in DEFAULT_CATEGORIES
    )
    ops_json = json.dumps(items, ensure_ascii=False, default=str)
    prompt = (
        "Ты — финансовый аналитик. Категоризируй банковские операции компании.\n\n"
        f"Доступные статьи финмодели:\n{categories_block}\n\n"
        f"Операции (direction: in — поступление, out — списание):\n{ops_json}\n\n"
        "Ответь ТОЛЬКО валидным JSON-массивом без пояснений и markdown:\n"
        '[{"idx": <индекс>, "code": "<код статьи>", "confidence": <0.0-1.0>}]\n'
        "Используй ТОЛЬКО коды из списка выше. Если не уверен — ставь confidence < 0.7."
    )
    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        response = model.generate_content(prompt)
        text = response.text.replace("```json", "").replace("```", "").strip()
        results = {}
        for row in json.loads(text):
            idx  = int(row["idx"])
            code = str(row.get("code", ""))
            conf = float(row.get("confidence", 0))
            if code in CAT_BY_CODE:
                results[idx] = (code, max(0.0, min(conf, 1.0)))
        return results
    except Exception as e:
        print(f"   ⚠️  Gemini error: {e}")
        return {}

# ── Level 4: Fallback ─────────────────────────────────────────────────────────
def _fallback(op: ParsedOperation) -> tuple:
    code = "REV_MAIN" if op.direction == "in" else "OTHER_EXP"
    return code, 0.4, "fallback"

# ── Agent 2: Categorization Agent ─────────────────────────────────────────────
def categorization_agent(ops: list[ParsedOperation]) -> list[CategorizedOperation]:
    """
    Agent 2 — Categorization Agent.
    4-level cascade: memory rules → keywords → Gemini AI → fallback.
    Evals: operations below confidence threshold are flagged for review.
    """
    print(f"\n🏷️  [Categorization Agent] Processing {len(ops)} operations...")
    results: list[Optional[CategorizedOperation]] = [None] * len(ops)
    ai_queue: dict[int, dict] = {}   # idx → item for Gemini batch

    # Levels 1 & 2: rules and keywords (fast, no API call)
    for i, op in enumerate(ops):
        hit = _apply_rules(op) or _apply_keywords(op)
        if hit:
            code, conf, src = hit
            cat = CAT_BY_CODE[code]
            results[i] = CategorizedOperation(
                op=op, category_code=code, category_name=cat["name"],
                kind=cat["kind"], confidence=conf, source=src,
                needs_review=conf < CONFIDENCE_THRESHOLD,
            )
        else:
            ai_queue[i] = {
                "idx": i, "direction": op.direction,
                "amount": round(op.amount, 2),
                "counterparty": op.counterparty,
                "description": op.description,
            }

    # Level 3: Gemini for remaining ops (batches of 40)
    if ai_queue:
        batch_size = 40
        items = list(ai_queue.values())
        ai_results: dict[int, tuple] = {}
        for start in range(0, len(items), batch_size):
            ai_results.update(_ai_categorize_batch(items[start:start+batch_size]))
        for i, item in ai_queue.items():
            op = ops[i]
            if i in ai_results:
                code, conf = ai_results[i]
                cat = CAT_BY_CODE[code]
                results[i] = CategorizedOperation(
                    op=op, category_code=code, category_name=cat["name"],
                    kind=cat["kind"], confidence=conf, source="gemini",
                    needs_review=conf < CONFIDENCE_THRESHOLD,
                )
            else:
                # Level 4: fallback
                code, conf, src = _fallback(op)
                cat = CAT_BY_CODE[code]
                results[i] = CategorizedOperation(
                    op=op, category_code=code, category_name=cat["name"],
                    kind=cat["kind"], confidence=conf, source=src,
                    needs_review=True,
                )

    cat_ops = [r for r in results if r is not None]

    # ── Eval: print categorization quality report ─────────────────────────────
    by_source = defaultdict(int)
    for co in cat_ops:
        by_source[co.source] += 1
    needs_review = sum(1 for co in cat_ops if co.needs_review)
    avg_conf = sum(co.confidence for co in cat_ops) / len(cat_ops) if cat_ops else 0

    print(f"   ✅ Categorized: {len(cat_ops)} operations")
    print(f"   📊 By source  : " + " | ".join(f"{src}: {cnt}" for src, cnt in sorted(by_source.items())))
    print(f"   🎯 Avg confidence: {avg_conf:.2%}")
    print(f"   ⚠️  Needs review  : {needs_review} operations (confidence < {CONFIDENCE_THRESHOLD:.0%})")
    return cat_ops

## 4. Agent 3 — Financial Model Agent
> **Tool use:** Builds P&L, cash flow, KPIs, and investment metrics (NPV/IRR).

In [ ]:
def _month_key(d: date) -> str:
    return d.strftime("%Y-%m")

def _npv(monthly_rate: float, flows: list) -> float:
    return sum(cf / (1 + monthly_rate) ** (i+1) for i, cf in enumerate(flows))

def _invest_metrics(flows: list, yearly_pct: float = 15.0) -> dict:
    if not flows or not any(flows):
        return {"npv": None, "irr_pct": None, "payback_months": None}
    monthly_rate = (1 + yearly_pct / 100) ** (1/12) - 1
    npv = round(_npv(monthly_rate, flows), 2)
    irr_pct = None
    low, high = -0.95, 5.0
    if _npv(low, flows) * _npv(high, flows) < 0:
        for _ in range(100):
            mid = (low + high) / 2
            (_high := high) if _npv(low, flows) * _npv(mid, flows) > 0 else None
            if _npv(low, flows) * _npv(mid, flows) <= 0: high = mid
            else: low = mid
        irr_pct = round(((1 + (low+high)/2)**12 - 1) * 100, 1)
    cumulative, payback, was_neg = 0.0, None, False
    for i, cf in enumerate(flows):
        cumulative += cf
        if cumulative < 0: was_neg = True
        elif was_neg and payback is None: payback = i + 1
    if not was_neg: payback = 0
    return {"npv": npv, "irr_pct": irr_pct, "payback_months": payback}

def financial_model_agent(cat_ops: list[CategorizedOperation]) -> dict:
    """
    Agent 3 — Financial Model Agent.
    Builds monthly P&L, cash flow, expense breakdown, and investment KPIs.
    Excludes transfers from P&L (they are balance sheet items).
    """
    print(f"\n📐 [Financial Model Agent] Building financial model...")

    # Aggregate by month and category
    monthly: dict[str, dict[str, float]] = defaultdict(lambda: defaultdict(float))
    for co in cat_ops:
        if co.kind == "transfer":
            continue   # exclude internal transfers from P&L
        month = _month_key(co.op.date)
        monthly[month][co.category_code] += co.op.amount

    months = sorted(monthly.keys())

    # Monthly P&L series
    income_codes  = {c["code"] for c in DEFAULT_CATEGORIES if c["kind"] == "income"}
    expense_codes = {c["code"] for c in DEFAULT_CATEGORIES if c["kind"] in ("expense", "investing", "financing")}

    revenue_series  = [sum(monthly[m].get(c, 0) for c in income_codes)  for m in months]
    expense_series  = [sum(monthly[m].get(c, 0) for c in expense_codes) for m in months]
    profit_series   = [r - e for r, e in zip(revenue_series, expense_series)]
    cashflow_series = profit_series   # simplified: CF ≈ profit (no accrual adjustments)
    cum_cf          = []
    running = 0.0
    for cf in cashflow_series:
        running += cf
        cum_cf.append(running)

    # KPIs
    total_revenue = sum(revenue_series)
    total_expense = sum(expense_series)
    total_profit  = total_revenue - total_expense
    margin_pct    = (total_profit / total_revenue * 100) if total_revenue else 0

    # Expense breakdown
    expense_breakdown: dict[str, float] = defaultdict(float)
    for co in cat_ops:
        if co.kind == "expense":
            expense_breakdown[co.category_name] += co.op.amount

    # Investment metrics
    invest_metrics = _invest_metrics(cashflow_series)

    model = {
        "months": months,
        "revenue_series": revenue_series,
        "expense_series": expense_series,
        "profit_series": profit_series,
        "cum_cashflow": cum_cf,
        "kpis": {
            "total_revenue": total_revenue,
            "total_expense": total_expense,
            "total_profit": total_profit,
            "margin_pct": margin_pct,
        },
        "expense_breakdown": dict(expense_breakdown),
        "invest_metrics": invest_metrics,
        "monthly_detail": {m: dict(d) for m, d in monthly.items()},
    }

    print(f"   ✅ Model built for {len(months)} months: {months[0] if months else '—'} → {months[-1] if months else '—'}")
    k = model["kpis"]
    print(f"   💰 Revenue : {k['total_revenue']:>12,.0f}")
    print(f"   💸 Expenses: {k['total_expense']:>12,.0f}")
    print(f"   📈 Profit  : {k['total_profit']:>12,.0f}  (margin {k['margin_pct']:.1f}%)")
    im = invest_metrics
    if im["npv"] is not None:
        print(f"   🏦 NPV: {im['npv']:,.0f} | IRR: {im['irr_pct']}% | Payback: {im['payback_months']} months")
    return model

## 5. Agent 4 — Insight Agent
> **Multi-agent handoff:** Receives model output and generates Gemini-powered business insights.

In [ ]:
def insight_agent(model: dict, cat_ops: list[CategorizedOperation]) -> str:
    """
    Agent 4 — Insight Agent.
    Analyzes the financial model and generates actionable business recommendations
    using Gemini. Falls back to rule-based insights if API is unavailable.
    """
    print(f"\n💡 [Insight Agent] Generating business insights...")
    k = model["kpis"]
    top_expenses = sorted(model["expense_breakdown"].items(), key=lambda x: -x[1])[:5]
    needs_review_count = sum(1 for co in cat_ops if co.needs_review)

    # Build context summary for Gemini
    context = f"""
Финансовые показатели бизнеса за {len(model['months'])} месяцев:
- Выручка: {k['total_revenue']:,.0f} руб.
- Расходы: {k['total_expense']:,.0f} руб.
- Прибыль: {k['total_profit']:,.0f} руб. (маржинальность {k['margin_pct']:.1f}%)
- Динамика прибыли по месяцам: {[round(p) for p in model['profit_series']]}
- Топ-5 статей расходов:
{chr(10).join(f'  • {name}: {amt:,.0f} руб.' for name, amt in top_expenses)}
- Операций требуют проверки категории: {needs_review_count}
- NPV: {model['invest_metrics'].get('npv')}
- IRR: {model['invest_metrics'].get('irr_pct')}%
"""
    if GEMINI_API_KEY:
        prompt = (
            "Ты — опытный финансовый советник для малого бизнеса. "
            "Проанализируй показатели и дай 4-5 конкретных рекомендаций.\n"
            "Структурируй ответ: проблемы (🔴), возможности (🟢), действия (📌).\n\n"
            + context
        )
        try:
            model_g = genai.GenerativeModel("gemini-2.0-flash")
            response = model_g.generate_content(prompt)
            insights = response.text
            print("   ✅ Gemini insights generated")
            return insights
        except Exception as e:
            print(f"   ⚠️  Gemini error: {e} — using rule-based insights")

    # Rule-based fallback insights
    lines = ["📊 Financial Health Summary\n"]
    if k["margin_pct"] < 10:
        lines.append("🔴 Низкая маржинальность (<10%). Проверьте структуру расходов.")
    elif k["margin_pct"] > 30:
        lines.append("🟢 Высокая маржинальность (>30%). Бизнес работает эффективно.")
    if top_expenses:
        top_name, top_amt = top_expenses[0]
        pct = top_amt / k["total_expense"] * 100 if k["total_expense"] else 0
        if pct > 40:
            lines.append(f"🔴 '{top_name}' составляет {pct:.0f}% расходов — высокая концентрация риска.")
    profits = model["profit_series"]
    if len(profits) > 1 and profits[-1] > profits[0]:
        lines.append("🟢 Положительная динамика прибыли — бизнес растёт.")
    elif len(profits) > 1 and profits[-1] < profits[0]:
        lines.append("🔴 Снижение прибыли — требуется анализ причин.")
    if needs_review_count > 0:
        lines.append(f"📌 {needs_review_count} операций требуют ручной проверки категорий.")
    lines.append("📌 Подключите данные за следующий месяц для уточнения прогноза.")
    return "\n".join(lines)

## 6. Visualization

In [ ]:
def plot_dashboard(model: dict):
    """Render 4-panel financial dashboard."""
    months = model["months"]
    if not months:
        print("No data to plot.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Financial Dashboard", fontsize=15, fontweight="bold", y=0.98)
    colors = {"revenue": "#2ecc71", "expense": "#e74c3c",
              "profit": "#3498db",  "cumcf": "#9b59b6"}

    fmt = mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k")
    x = range(len(months))
    short_months = [m[5:] + "/" + m[2:4] for m in months]

    # Panel 1: Revenue vs Expenses
    ax = axes[0, 0]
    ax.bar(x, model["revenue_series"], label="Revenue",  color=colors["revenue"], alpha=0.8)
    ax.bar(x, model["expense_series"], label="Expenses", color=colors["expense"], alpha=0.6)
    ax.set_title("Revenue vs Expenses"); ax.set_xticks(list(x)); ax.set_xticklabels(short_months, rotation=45)
    ax.yaxis.set_major_formatter(fmt); ax.legend(); ax.grid(axis="y", alpha=0.3)

    # Panel 2: Monthly Profit
    ax = axes[0, 1]
    bar_colors = [colors["profit"] if p >= 0 else colors["expense"] for p in model["profit_series"]]
    ax.bar(x, model["profit_series"], color=bar_colors, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Monthly Profit / Loss"); ax.set_xticks(list(x)); ax.set_xticklabels(short_months, rotation=45)
    ax.yaxis.set_major_formatter(fmt); ax.grid(axis="y", alpha=0.3)

    # Panel 3: Cumulative Cash Flow
    ax = axes[1, 0]
    ax.plot(list(x), model["cum_cashflow"], color=colors["cumcf"], linewidth=2.5, marker="o", markersize=5)
    ax.fill_between(list(x), model["cum_cashflow"], alpha=0.15, color=colors["cumcf"])
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Cumulative Cash Flow"); ax.set_xticks(list(x)); ax.set_xticklabels(short_months, rotation=45)
    ax.yaxis.set_major_formatter(fmt); ax.grid(alpha=0.3)

    # Panel 4: Expense Breakdown (pie)
    ax = axes[1, 1]
    breakdown = model["expense_breakdown"]
    if breakdown:
        top = sorted(breakdown.items(), key=lambda x: -x[1])[:7]
        labels = [f"{n[:18]}" for n, _ in top]
        values = [v for _, v in top]
        wedges, texts, autotexts = ax.pie(
            values, labels=labels, autopct="%1.0f%%",
            startangle=140, pctdistance=0.8,
            textprops={"fontsize": 7.5}
        )
        for at in autotexts: at.set_fontsize(7)
    ax.set_title("Expense Breakdown")

    plt.tight_layout()
    plt.savefig("financial_dashboard.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("   📊 Dashboard saved: financial_dashboard.png")

## 7. Main Pipeline — Run the Agent
> Upload your bank statement CSV to Kaggle (Input → Upload) and update `FILE_PATH` below.

In [ ]:
DEMO_CSV = """Дата;Тип операции;Сумма;Контрагент;Назначение платежа
12.01.2026;Поступление;280000;ООО Альфа Трейд;Оплата по счету № 12 за услуги
15.01.2026;Поступление;150000;ИП Смирнов А.В.;Оплата по договору № 7 за консультационные услуги
22.01.2026;Поступление;270000;ООО Омега Групп;Предоплата по договору № 15 за поставку
05.01.2026;Списание;85000;ООО БизнесПарк;Аренда офиса за январь 2026
10.01.2026;Списание;120000;Сотрудники по реестру;Зарплата за вторую половину декабря по реестру № 1
25.01.2026;Списание;125000;Сотрудники по реестру;Аванс сотрудникам за январь по реестру № 2
20.01.2026;Списание;42000;ИФНС № 7;ЕНС единый налоговый платеж УСН за 4 квартал
28.01.2026;Списание;30000;ООО Яндекс;Оплата рекламной кампании Яндекс Директ
31.01.2026;Списание;1990;АО Банк Точка;Комиссия за обслуживание счета за январь
18.01.2026;Списание;95000;ООО СнабСервис;Закупка товара по счету № 88
26.01.2026;Списание;21200;ООО ЭкоКлин;Клининг офисного помещения январь 2026
10.02.2026;Поступление;320000;ООО Альфа Трейд;Оплата по счету № 18 за услуги
14.02.2026;Поступление;95000;ИП Петров М.С.;Предоплата по договору № 9
20.02.2026;Поступление;285000;ООО Омега Групп;Оплата по счету № 21 за поставку
05.02.2026;Списание;85000;ООО БизнесПарк;Аренда офиса за февраль 2026
20.02.2026;Списание;245000;Сотрудники по реестру;Зарплата за январь по реестру № 3
22.02.2026;Списание;52000;ИФНС № 7;Налог на прибыль за 4 квартал
25.02.2026;Списание;35000;ООО Яндекс;Реклама Яндекс Директ февраль
28.02.2026;Списание;1990;АО Банк Точка;Комиссия за обслуживание счета за февраль
15.02.2026;Списание;110000;ООО СнабСервис;Закупка материалов по счету № 102
10.03.2026;Поступление;410000;ООО Бета Групп;Оплата по счету № 25 за товары
12.03.2026;Поступление;180000;ООО Альфа Трейд;Оплата по счету № 22 за услуги
19.03.2026;Поступление;210000;ООО Омега Групп;Оплата по счету № 28 за поставку
05.03.2026;Списание;85000;ООО БизнесПарк;Аренда офиса за март 2026
20.03.2026;Списание;245000;Сотрудники по реестру;Зарплата за февраль по реестру № 4
25.03.2026;Списание;28000;ИФНС № 7;Страховые взносы за февраль
28.03.2026;Списание;40000;ООО Медиаплан;Маркетинг и продвижение март
31.03.2026;Списание;1990;АО Банк Точка;Комиссия за обслуживание счета за март
15.03.2026;Списание;140000;ООО СнабСервис;Закупка товара по счету № 118
20.03.2026;Списание;65000;ООО ИТ Решения;Лицензия CRM система годовая
22.03.2026;Списание;41000;ООО СберЛизинг;Лизинговый платеж за оборудование март
10.04.2026;Поступление;380000;ООО Гамма;Оплата по счету № 31 за консультации
18.04.2026;Поступление;220000;ИП Смирнов А.В.;Оплата по договору № 11 за услуги
24.04.2026;Поступление;300000;ООО Омега Групп;Оплата по счету № 35 за поставку
05.04.2026;Списание;85000;ООО БизнесПарк;Аренда офиса за апрель 2026
20.04.2026;Списание;260000;Сотрудники по реестру;Зарплата за март по реестру № 5
22.04.2026;Списание;31000;ИФНС № 7;Страховые взносы за март
28.04.2026;Списание;45000;ООО Медиаплан;Маркетинг апрель
30.04.2026;Списание;1990;АО Банк Точка;Комиссия за обслуживание счета за апрель
15.04.2026;Списание;125000;ООО СнабСервис;Закупка материалов по счету № 135
10.05.2026;Поступление;450000;ООО Дельта Трейд;Оплата по счету № 38 за товары
15.05.2026;Поступление;150000;ООО Альфа Трейд;Оплата по счету № 35 за услуги
23.05.2026;Поступление;400000;ООО Омега Групп;Оплата по счету № 42 за поставку
05.05.2026;Списание;85000;ООО БизнесПарк;Аренда офиса за май 2026
20.05.2026;Списание;265000;Сотрудники по реестру;Зарплата за апрель по реестру № 6
22.05.2026;Списание;33000;ИФНС № 7;Страховые взносы за апрель
28.05.2026;Списание;50000;ООО Медиаплан;Маркетинг май - таргетированная реклама
31.05.2026;Списание;1990;АО Банк Точка;РКО комиссия за май
15.05.2026;Списание;155000;ООО СнабСервис;Закупка товара по счету № 152
29.05.2026;Поступление;47500;АО Техносфера;Возврат переплаты по акту сверки № 7
27.05.2026;Списание;36000;ИП Коваленко Д.Р.;Курьерская доставка заказов клиентам май
22.01.2026;Списание;18700;ООО СберЛизинг;Лизинговый платеж за оборудование январь
22.02.2026;Списание;18700;ООО СберЛизинг;Лизинговый платеж за оборудование февраль
17.03.2026;Списание;28500;ООО Вертикаль;Счет № 77 от 15.03.2026
14.04.2026;Поступление;85000;ИП Захарова Н.С.;Договор № 3А от 01.04.2026
25.04.2026;Списание;19800;ООО Форс Мажор;Счет № 19 апрель 2026
18.05.2026;Списание;44000;ИП Воронов А.А.;Счет № 56 от 17.05.2026
"""

with open("demo_statement.csv", "w", encoding="utf-8") as f:
    f.write(DEMO_CSV)
print("✅ Demo file saved: demo_statement.csv")
print("   Change FILE_PATH below to use your own bank statement.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — set your file path here
# ─────────────────────────────────────────────────────────────────────────────
FILE_PATH = "demo_statement.csv"   # ← change to your uploaded file path

# Optional: pre-seed the agent memory with known counterparties
# This simulates a business that has already been using the system
teach_rule("АО Банк Точка", "BANK")
teach_rule("ООО БизнесПарк", "RENT")
teach_rule("ООО Медиаплан", "MARKETING")

print(f"\n{'='*60}")
print(f"  💼 Financial Statement Agent — Starting Pipeline")
print(f"{'='*60}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN THE MULTI-AGENT PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

# Agent 1: Parse
parsed_ops = parsing_agent(FILE_PATH)

# Agent 2: Categorize
cat_ops = categorization_agent(parsed_ops)

# Agent 3: Build financial model
fin_model = financial_model_agent(cat_ops)

# Agent 4: Generate insights
insights = insight_agent(fin_model, cat_ops)

print(f"\n{'='*60}")
print("  ✅ Pipeline complete")
print(f"{'='*60}")

In [ ]:
# Dashboard
plot_dashboard(fin_model)

In [ ]:
# Insights
print("\n" + "="*60)
print("  💡 Business Insights")
print("="*60)
print(insights)

In [ ]:
# Categorized transactions table (top 20)
print("\n📋 Categorized Transactions (sample):")
rows = []
for co in cat_ops[:25]:
    rows.append({
        "Date": co.op.date,
        "Direction": "📥" if co.op.direction == "in" else "📤",
        "Amount": f"{co.op.amount:,.0f}",
        "Counterparty": co.op.counterparty[:30],
        "Category": co.category_name[:25],
        "Confidence": f"{co.confidence:.0%}",
        "Source": co.source,
        "Review?": "⚠️" if co.needs_review else "✅",
    })
df_display = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 35)
pd.set_option("display.width", 120)
print(df_display.to_string(index=False))

## 8. Agent Memory — Teach a New Rule
> Demonstrate the memory system: confirm a category to create a learned rule.

In [ ]:
# Show ops that need review
review_ops = [co for co in cat_ops if co.needs_review]
print(f"\n⚠️  Operations needing review: {len(review_ops)}")
for i, co in enumerate(review_ops[:5]):
    print(f"  [{i}] {co.op.date} | {co.op.amount:>10,.0f} {co.op.direction} "
          f"| {co.op.counterparty[:30]:<30} → {co.category_name} ({co.confidence:.0%} {co.source})")

# Teach the agent a new rule (simulates user confirming a category)
if review_ops:
    chosen = review_ops[0]
    print(f"\n🧠 Teaching rule for: '{chosen.op.counterparty}'")
    teach_rule(chosen.op.counterparty, chosen.category_code)
    print(f"   Next time '{chosen.op.counterparty}' appears → auto-categorized as '{chosen.category_name}'")
    print(f"   Total learned rules: {len(learned_rules)}")

---
## Summary

This notebook demonstrates a **4-agent financial pipeline** built on the [data_fin_model](https://github.com/Olegmeln/data_fin_model) codebase:

| Agent | Role | Course Concept |
|---|---|---|
| **Parsing Agent** | Auto-detects and parses CSV/Excel/1C bank statements | Tool use |
| **Categorization Agent** | 4-level cascade: rules → keywords → Gemini → fallback | Multi-agent + Memory |
| **Financial Model Agent** | Builds P&L, cash flow, NPV/IRR | Tool use |
| **Insight Agent** | Gemini-powered business recommendations | Multi-agent |

**Production deployment:** https://data-fin-model.vercel.app  
**Repository:** https://github.com/Olegmeln/data_fin_model